# ✈️ Airline Passenger Satisfaction — Dual-T4 GPU Ensemble
### XGBoost · CatBoost · Neural Net · Pairwise Target Encoding · Stacking  —  Playground Series S6E10

**Task.** Predict the probability that a passenger is `satisfied`. Submissions are scored with **ROC-AUC**.

**Approach (competition data only).**

| Step | What | Why |
|---|---|---|
| 1 | Light EDA | Confirm data shape, missing values, which features carry signal |
| 2 | Feature engineering | Rating aggregates, delay features, `Class × Type of Travel` |
| 3 | Pairwise target encoding (computed *inside* each CV fold) | Hands the trees the strongest feature interactions directly |
| 4 | Optuna-tuned **XGBoost** on GPU (two feature sets) | Strong, fast GBDT; tuning trials run on both GPUs at once |
| 5 | **CatBoost** on **both GPUs** (multi-GPU training) | Excels when most features are low-cardinality categoricals — like 0–5 ratings |
| 6 | **MLP with embeddings** (PyTorch) | Different inductive bias → diversity for the ensemble |
| 7 | Stacking on out-of-fold predictions | Learns how much to trust each model |

**Accelerator:** set **GPU T4 ×2** in *Settings → Accelerator*. Every GPU stage below is written to keep **both** T4s busy:
XGBoost / MLP folds are dispatched two-at-a-time (one per GPU), Optuna runs two trials in parallel (one per GPU),
and CatBoost trains each fold across both GPUs with `devices='0:1'`.

> Expected runtime with the default settings: roughly **1–2 h** on GPU T4 ×2 (CatBoost is the longest stage).

## 0. Setup
Imports and global configuration. `ON_KAGGLE` switches on the full run; anywhere else the notebook runs a fast
**debug** mode (small subsample, few folds/trials) so the whole pipeline can be smoke-tested on a CPU.

In [ ]:
import os, glob, time, queue, contextlib, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import catboost as cb
import optuna
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.special import logit
from scipy.stats import rankdata

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 60)

ON_KAGGLE = os.path.exists('/kaggle/input')
DEBUG = not ON_KAGGLE

SEED = 42
TARGET = 'satisfaction'
N_FOLDS = 3 if DEBUG else 10

print('xgboost', xgb.__version__, '| catboost', cb.__version__, '| torch', torch.__version__, '| optuna', optuna.__version__)
print('ON_KAGGLE =', ON_KAGGLE, '| DEBUG =', DEBUG)

### GPU discovery and the two-GPU work pool
`N_GPUS` is read from PyTorch. A small **GPU pool** (a thread-safe queue holding the GPU ids) hands out one GPU per job:
a training job borrows a GPU id, trains on it, and returns it. Running jobs in a 2-thread pool therefore keeps
**GPU 0 and GPU 1 busy simultaneously**. XGBoost, CatBoost and PyTorch all release Python's GIL while they compute,
so threads give real parallelism here.

In [ ]:
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}  {p.total_memory / 1e9:.1f} GB')
if N_GPUS == 0:
    print('No GPU found -> running on CPU (fine for DEBUG; on Kaggle choose Accelerator = GPU T4 x2)')
if ON_KAGGLE and N_GPUS < 2:
    print('WARNING: fewer than 2 GPUs visible - select "GPU T4 x2" in the notebook settings for full speed.')

N_WORKERS = max(N_GPUS, 1)          # parallel jobs = one per GPU
GPU_POOL = queue.Queue()
for g in range(N_WORKERS):
    GPU_POOL.put(g)

@contextlib.contextmanager
def gpu_slot():
    """Borrow a GPU id from the pool for the duration of a job."""
    g = GPU_POOL.get()
    try:
        yield g
    finally:
        GPU_POOL.put(g)

def xgb_device(g):
    return f'cuda:{g}' if N_GPUS else 'cpu'

def torch_device(g):
    return torch.device(f'cuda:{g}' if N_GPUS else 'cpu')

## 1. Load data
Kaggle mounts competition data under `/kaggle/input/`. The path is discovered with a glob so the notebook works whatever
the exact mount folder is called. Outputs go to `/kaggle/working/` (the notebook's *Output* tab).

In [ ]:
if ON_KAGGLE:
    hits = [p for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
            if os.path.exists(os.path.join(os.path.dirname(p), 'sample_submission.csv'))]
    DATA_DIR = os.path.dirname(hits[0])
    OUT_DIR = '/kaggle/working'
else:
    DATA_DIR = os.environ.get('AIRLINE_DATA', '.')
    OUT_DIR = '.'
print('Data dir:', DATA_DIR)

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

if DEBUG:   # small slice for a quick CPU smoke test
    train = train.sample(30_000, random_state=SEED).reset_index(drop=True)
    test = test.sample(10_000, random_state=SEED).reset_index(drop=True)
    sample = sample.set_index('id').loc[test['id']].reset_index()

train[TARGET] = train[TARGET].astype(str).str.lower().eq('true')   # robust to bool or 'True'/'False' strings
y = train[TARGET].astype(int).values
print('train', train.shape, '| test', test.shape)
train.head()

## 2. Quick EDA
The 13 service ratings are on a 0–5 scale where **0 means "not applicable"**. Four columns are categorical,
four are numeric (age, distance, two delays). We check missing values, class balance and the single-feature AUC
of each column to see where the signal is.

In [ ]:
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATINGS = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location',
           'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service',
           'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RAW = NUMS + RATINGS + CATS

print(f'Satisfied rate: {y.mean():.3f}')
display(pd.DataFrame({'n_unique': train[RAW].nunique(),
                      'train_missing': train[RAW].isna().sum(),
                      'test_missing': test[RAW].isna().sum()}).T)

uni = {}
for c in RAW:
    x = train[c].map(train.groupby(c)[TARGET].mean()) if c in CATS else train[c].fillna(-1)
    a = roc_auc_score(y, x)
    uni[c] = max(a, 1 - a)
uni = pd.Series(uni).sort_values()

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(uni.index, uni.values - 0.5, left=0.5, color='#2a78d6', height=0.6)
for i, v in enumerate(uni.values):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=8, color='#52514e')
ax.set_xlim(0.5, 0.9); ax.set_xlabel('ROC-AUC'); ax.set_title('Single-feature ROC-AUC')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

**Takeaways.** *Online boarding*, *Inflight wifi*, *Class* and *Type of Travel* dominate. Ratings behave like ordered
categories, delays are weak on their own, and *Arrival Delay* has a handful of missing values (handled natively by the
tree models, imputed for the neural net).

## 3. Feature engineering
Row-level features shared by all models (no target information, so safe to compute on train and test together):

* **Rating aggregates** — mean / std / min / max over the ratings (with 0 = N/A treated as missing), number of N/A
  ratings, number of 5-star ratings.
* **Theme scores** — a *digital* score (wifi, online boarding, online booking) and a *comfort* score (seat, entertainment,
  leg room, cleanliness).
* **Delay features** — arrival minus departure delay, total delay, delay per km.
* **`class_travel`** — the `Class × Type of Travel` combination, the single strongest segmentation in the data.

In [ ]:
def make_features(df):
    out = df[RAW].copy()
    r = out[RATINGS].replace(0, np.nan)
    out['rating_mean'] = r.mean(axis=1)
    out['rating_std'] = r.std(axis=1)
    out['rating_min'] = r.min(axis=1)
    out['rating_max'] = r.max(axis=1)
    out['n_na_ratings'] = (out[RATINGS] == 0).sum(axis=1)
    out['n_5_ratings'] = (out[RATINGS] == 5).sum(axis=1)
    out['digital_score'] = out[['Inflight wifi service', 'Online boarding', 'Ease of Online booking']].mean(axis=1)
    out['comfort_score'] = out[['Seat comfort', 'Inflight entertainment', 'Leg room service', 'Cleanliness']].mean(axis=1)
    dep, arr = out['Departure Delay in Minutes'], out['Arrival Delay in Minutes']
    out['delay_diff'] = arr - dep
    out['total_delay'] = arr.fillna(dep) + dep
    out['delay_per_km'] = out['total_delay'] / (out['Flight Distance'] + 1)
    out['class_travel'] = out['Class'].astype(str) + '_' + out['Type of Travel'].astype(str)
    return out

full = make_features(pd.concat([train, test], ignore_index=True))
CAT_COLS = CATS + ['class_travel']
for c in CAT_COLS:
    full[c] = full[c].astype(str).astype('category')

n_train = len(train)
X = full.iloc[:n_train].reset_index(drop=True)
X_test = full.iloc[n_train:].reset_index(drop=True)
FEATURES = list(X.columns)
print(len(FEATURES), 'features')

## 4. Pairwise target encoding
Satisfaction depends strongly on *combinations* (e.g. a business traveller in Business class with poor online boarding).
Trees can find these, but it takes many splits. Instead we give them the answer directly: for every pair (and a few
triples) of the most informative columns we compute the **smoothed satisfaction rate of that combination**.

**Leakage control.** Target encodings must never see the row they encode:
* For the **validation** fold and the **test** set, the encoding is fitted on the whole training part of the fold.
* For the **training** part, the encoding is *cross-fitted* with an inner 5-fold split (each row is encoded by a mapping
  learned on the other 4/5).

Every column is first mapped to a small integer code, so a pair is a single integer key and an encoding is just two
`np.bincount` calls — fast enough to recompute inside each of the CV folds.

In [ ]:
TE_COLS = ['Online boarding', 'Inflight wifi service', 'Class', 'Type of Travel', 'Customer Type',
           'Inflight entertainment', 'Seat comfort', 'Ease of Online booking', 'Leg room service', 'On-board service']
TE_TRIPLES = [('Class', 'Type of Travel', 'Online boarding'),
              ('Class', 'Type of Travel', 'Inflight wifi service'),
              ('Customer Type', 'Type of Travel', 'Online boarding'),
              ('Online boarding', 'Inflight wifi service', 'Type of Travel')]
TE_SMOOTH = 20.0

codes = {c: pd.factorize(full[c].astype(str), sort=True)[0].astype(np.int64) for c in TE_COLS}

def combo_key(cols):
    key = np.zeros(len(full), dtype=np.int64)
    for c in cols:
        key = key * (codes[c].max() + 1) + codes[c]
    return pd.factorize(key)[0]           # compact 0..k-1 ids

TE_GROUPS = [(a, b) for i, a in enumerate(TE_COLS) for b in TE_COLS[i + 1:]] + TE_TRIPLES
TE_KEYS = np.stack([combo_key(g) for g in TE_GROUPS], axis=1).astype(np.int32)   # (n_train + n_test, n_groups)
TE_NKEYS = TE_KEYS.max(axis=0) + 1
TE_NAMES = ['te__' + '__'.join(g) for g in TE_GROUPS]
KEYS_TR, KEYS_TE = TE_KEYS[:n_train], TE_KEYS[n_train:]
print(len(TE_GROUPS), 'target-encoded combinations')

def _encode(keys, yy, nk):
    prior = yy.mean()
    s = np.bincount(keys, weights=yy, minlength=nk)
    c = np.bincount(keys, minlength=nk)
    return ((s + TE_SMOOTH * prior) / (c + TE_SMOOTH)).astype(np.float32)

def te_features(tr_idx, va_idx, seed):
    """Leak-free target encodings for one outer CV fold -> (train_part, valid_part, test) frames."""
    ytr = y[tr_idx].astype(np.float64)
    inner = list(KFold(5, shuffle=True, random_state=seed).split(tr_idx))
    F_tr = np.empty((len(tr_idx), len(TE_GROUPS)), np.float32)
    F_va = np.empty((len(va_idx), len(TE_GROUPS)), np.float32)
    F_te = np.empty((len(KEYS_TE), len(TE_GROUPS)), np.float32)
    for j, nk in enumerate(TE_NKEYS):
        k_tr = KEYS_TR[tr_idx, j]
        full_map = _encode(k_tr, ytr, nk)
        F_va[:, j] = full_map[KEYS_TR[va_idx, j]]
        F_te[:, j] = full_map[KEYS_TE[:, j]]
        for a, b in inner:
            F_tr[b, j] = _encode(k_tr[a], ytr[a], nk)[k_tr[b]]
    mk = lambda F: pd.DataFrame(F, columns=TE_NAMES)
    return mk(F_tr), mk(F_va), mk(F_te)

## 5. Cross-validation engine (two GPUs in parallel)
`run_cv` trains one model type over all folds (optionally several seeds, each with a different fold split) and returns:
* **OOF predictions** — every training row predicted by a model that never saw it → honest AUC estimate and the input to stacking;
* **test predictions** — averaged over all fold models.

Jobs are submitted to a thread pool with one worker per GPU; each job borrows a GPU from the pool, so with T4 ×2 two
folds always train at the same time. OOF / test predictions are saved to `OUT_DIR` as `.npy` so a long run's partial
results are never lost.

In [ ]:
def make_folds(seed):
    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(X, y))

def run_cv(name, fit_fn, n_seeds=1, parallel=True):
    oof = np.zeros(n_train)
    pred = np.zeros(len(X_test))
    jobs = [(s, k, tr, va) for s in range(n_seeds) for k, (tr, va) in enumerate(make_folds(SEED + s))]

    def work(job):
        s, k, tr, va = job
        with gpu_slot() as g:
            t0 = time.time()
            va_pred, te_pred = fit_fn(tr, va, SEED + s, g)
        print(f'  [{name}] seed {s} fold {k} on GPU {g}: AUC {roc_auc_score(y[va], va_pred):.5f}  ({time.time() - t0:.0f}s)', flush=True)
        return va, va_pred, te_pred

    t0 = time.time()
    with ThreadPoolExecutor(N_WORKERS if parallel else 1) as ex:
        for va, va_pred, te_pred in ex.map(work, jobs):
            oof[va] += va_pred / n_seeds
            pred += te_pred / (n_seeds * N_FOLDS)

    score = roc_auc_score(y, oof)
    print(f'>>> {name}: OOF AUC {score:.5f}   ({(time.time() - t0) / 60:.1f} min)')
    np.save(f'{OUT_DIR}/oof_{name}.npy', oof)
    np.save(f'{OUT_DIR}/test_{name}.npy', pred)
    return dict(oof=oof, pred=pred, score=score)

results = {}

## 6. XGBoost on GPU
### 6.1 Hyper-parameter search with Optuna (two trials at a time, one per GPU)
We tune on a single 80/20 split using the richest feature set (base + target encodings). Each trial borrows a GPU from
the pool, and `n_jobs=N_WORKERS` lets Optuna run two trials concurrently. A slightly higher learning rate (0.05) keeps
trials quick; the final models use 0.03 with early stopping, which only helps.

*Set `TUNE = False` to skip the search and use the defaults (saves ~15 min).*

In [ ]:
TUNE = True
N_TRIALS = 3 if DEBUG else 40

XGB_BASE = dict(objective='binary:logistic', eval_metric='auc', tree_method='hist',
                max_cat_to_onehot=1, max_bin=256, verbosity=0)
XGB_DEFAULT = dict(max_depth=7, min_child_weight=5, subsample=0.8, colsample_bytree=0.5,
                   reg_lambda=2.0, reg_alpha=0.1, gamma=0.0)

def xgb_train(params, Xtr, ytr, Xva, yva, g, lr, rounds=20000, es=300):
    p = {**XGB_BASE, **params, 'learning_rate': lr, 'device': xgb_device(g)}
    dtr = xgb.DMatrix(Xtr, ytr, enable_categorical=True)
    dva = xgb.DMatrix(Xva, yva, enable_categorical=True)
    m = xgb.train(p, dtr, rounds, evals=[(dva, 'va')], early_stopping_rounds=es, verbose_eval=False)
    return m, dva

# fixed tuning split + its target encodings (computed once, shared by all trials)
tune_tr, tune_va = make_folds(SEED)[0] if N_FOLDS >= 5 else next(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y))
_tr, _va, _ = te_features(tune_tr, tune_va, SEED)
TUNE_XTR = pd.concat([X.iloc[tune_tr].reset_index(drop=True), _tr], axis=1)
TUNE_XVA = pd.concat([X.iloc[tune_va].reset_index(drop=True), _va], axis=1)

def objective(trial):
    params = dict(
        max_depth=trial.suggest_int('max_depth', 4, 10),
        min_child_weight=trial.suggest_float('min_child_weight', 1, 50, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 0.9),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-2, 20, log=True),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 5, log=True),
        gamma=trial.suggest_float('gamma', 0, 2),
    )
    with gpu_slot() as g:
        m, dva = xgb_train(params, TUNE_XTR, y[tune_tr], TUNE_XVA, y[tune_va], g, lr=0.05,
                           rounds=200 if DEBUG else 5000, es=150)
    return m.best_score

XGB_PARAMS = dict(XGB_DEFAULT)
if TUNE:
    t0 = time.time()
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.enqueue_trial(XGB_DEFAULT)                       # make sure the defaults are a candidate
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_WORKERS)
    XGB_PARAMS.update(study.best_params)
    print(f'Best holdout AUC {study.best_value:.5f} in {(time.time() - t0) / 60:.1f} min')
    display(study.trials_dataframe().sort_values('value', ascending=False).head(8))
print('XGB params:', XGB_PARAMS)

### 6.2 XGBoost — base features
Native categorical handling (`enable_categorical=True`) on the engineered feature set. Two seeds (two different fold
splits) are averaged for stability; folds run two-at-a-time across the GPUs.

In [ ]:
XGB_LR = 0.03
XGB_SEEDS = 1 if DEBUG else 2

def fit_xgb_base(tr, va, seed, g):
    m, dva = xgb_train({**XGB_PARAMS, 'seed': seed}, X.iloc[tr], y[tr], X.iloc[va], y[va], g, lr=XGB_LR,
                       rounds=300 if DEBUG else 20000)
    rng = (0, m.best_iteration + 1)
    return (m.predict(dva, iteration_range=rng),
            m.predict(xgb.DMatrix(X_test, enable_categorical=True), iteration_range=rng))

results['xgb_base'] = run_cv('xgb_base', fit_xgb_base, n_seeds=XGB_SEEDS)

### 6.3 XGBoost — base + pairwise target encodings
Same model, plus the fold-safe target encodings from section 4 (rebuilt inside every fold).

In [ ]:
def fit_xgb_te(tr, va, seed, g):
    f_tr, f_va, f_te = te_features(tr, va, seed)
    Xtr = pd.concat([X.iloc[tr].reset_index(drop=True), f_tr], axis=1)
    Xva = pd.concat([X.iloc[va].reset_index(drop=True), f_va], axis=1)
    Xte = pd.concat([X_test, f_te], axis=1)
    m, dva = xgb_train({**XGB_PARAMS, 'seed': seed}, Xtr, y[tr], Xva, y[va], g, lr=XGB_LR,
                       rounds=300 if DEBUG else 20000)
    rng = (0, m.best_iteration + 1)
    return m.predict(dva, iteration_range=rng), m.predict(xgb.DMatrix(Xte, enable_categorical=True), iteration_range=rng)

results['xgb_te'] = run_cv('xgb_te', fit_xgb_te, n_seeds=XGB_SEEDS)

## 7. CatBoost on both GPUs
CatBoost builds its own ordered target statistics and feature combinations for categorical columns, so we declare the
**13 ratings as categorical** as well as the 5 true categoricals. On GPU, `devices='0:1'` splits each model's training
across **both T4s**, so folds run one after another (`parallel=False`) with every fold using both GPUs.

Early stopping uses `Logloss`, which is computed natively on the GPU (`AUC` would be evaluated on the CPU and slow
training down); the fold AUC is computed afterwards.

In [ ]:
CB_CATS = RATINGS + CAT_COLS

def to_cb(df):
    out = df.copy()
    for c in CB_CATS:
        out[c] = out[c].astype(str)
    return out

X_cb, X_test_cb = to_cb(X), to_cb(X_test)
CB_PARAMS = dict(
    iterations=300 if DEBUG else 20000, learning_rate=0.06, depth=8, l2_leaf_reg=3.0,
    border_count=254, random_strength=1.0, bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', eval_metric='Logloss', od_type='Iter', od_wait=300,
    use_best_model=True, verbose=0,
)
if N_GPUS:
    CB_PARAMS.update(task_type='GPU', devices=':'.join(str(i) for i in range(N_GPUS)))   # '0:1' on T4 x2
else:
    CB_PARAMS.update(task_type='CPU', thread_count=-1)

def fit_cb(tr, va, seed, g):
    m = cb.CatBoostClassifier(**CB_PARAMS, random_seed=seed)
    m.fit(cb.Pool(X_cb.iloc[tr], y[tr], cat_features=CB_CATS),
          eval_set=cb.Pool(X_cb.iloc[va], y[va], cat_features=CB_CATS))
    return (m.predict_proba(X_cb.iloc[va])[:, 1],
            m.predict_proba(cb.Pool(X_test_cb, cat_features=CB_CATS))[:, 1])

results['catboost'] = run_cv('catboost', fit_cb, n_seeds=1, parallel=False)

## 8. Neural network (PyTorch MLP with embeddings)
Tree models dominate tabular data, but a neural net makes **different mistakes**, which is exactly what an ensemble
needs.

* **Embeddings** — each rating and categorical column gets a learned 8-dim vector per level.
* **Numeric inputs** — log-transformed distance/delays and the rating aggregates, standardised; missing values filled
  and flagged.
* **Training** — 3-layer MLP (512-256-128, SiLU, BatchNorm, dropout), AdamW + one-cycle LR schedule, best epoch kept by
  validation AUC.
* The whole fold lives on the GPU (no DataLoader), and folds run two-at-a-time, one per GPU.

In [ ]:
EMB_COLS = RATINGS + CAT_COLS
NN_NUM = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'rating_mean',
          'rating_std', 'rating_min', 'rating_max', 'n_na_ratings', 'n_5_ratings', 'digital_score', 'comfort_score',
          'delay_diff']

def nn_arrays(df_all):
    cat = np.stack([pd.factorize(df_all[c].astype(str), sort=True)[0] for c in EMB_COLS], axis=1).astype(np.int64)
    num = df_all[NN_NUM].astype(float).copy()
    for c in ['Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']:
        num[c] = np.log1p(num[c])
    num['delay_diff'] = np.sign(num['delay_diff']) * np.log1p(num['delay_diff'].abs())
    num['arr_missing'] = df_all['Arrival Delay in Minutes'].isna().astype(float)
    num = num.fillna(num.median())
    num = (num - num.mean()) / (num.std() + 1e-6)
    return cat, num.values.astype(np.float32)

NN_CAT, NN_NUMS = nn_arrays(full)
NN_CARD = NN_CAT.max(axis=0) + 1
NN_CAT_TR, NN_CAT_TE = NN_CAT[:n_train], NN_CAT[n_train:]
NN_NUM_TR, NN_NUM_TE = NN_NUMS[:n_train], NN_NUMS[n_train:]

class MLP(nn.Module):
    def __init__(self, cards, n_num, emb_dim=8, hidden=(512, 256, 128), drop=0.25):
        super().__init__()
        self.embs = nn.ModuleList(nn.Embedding(int(c), emb_dim) for c in cards)
        layers, d = [], emb_dim * len(cards) + n_num
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(drop)]
            d = h
        layers.append(nn.Linear(d, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, xc, xn):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.mlp(torch.cat(e + [xn], dim=1)).squeeze(1)

NN_EPOCHS = 2 if DEBUG else 14
NN_BATCH = 2048

@torch.no_grad()
def nn_predict(model, xc, xn, bs=65536):
    model.eval()
    return np.concatenate([torch.sigmoid(model(xc[i:i + bs], xn[i:i + bs])).float().cpu().numpy()
                           for i in range(0, len(xc), bs)])

def fit_nn(tr, va, seed, g):
    dev = torch_device(g)
    gen = torch.Generator(device='cpu').manual_seed(seed)
    to = lambda a: torch.as_tensor(a, device=dev)
    xc_tr, xn_tr, y_tr = to(NN_CAT_TR[tr]), to(NN_NUM_TR[tr]), to(y[tr].astype(np.float32))
    xc_va, xn_va = to(NN_CAT_TR[va]), to(NN_NUM_TR[va])

    torch.manual_seed(seed)
    model = MLP(NN_CARD, NN_NUM_TR.shape[1]).to(dev)
    steps = len(tr) // NN_BATCH            # drop the last partial batch (BatchNorm needs >1 row)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=NN_EPOCHS * steps, pct_start=0.25)
    loss_fn = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None
    for epoch in range(NN_EPOCHS):
        model.train()
        perm = torch.randperm(len(tr), generator=gen).to(dev)
        for i in range(steps):
            idx = perm[i * NN_BATCH:(i + 1) * NN_BATCH]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xc_tr[idx], xn_tr[idx]), y_tr[idx])
            loss.backward()
            opt.step(); sched.step()
        auc = roc_auc_score(y[va], nn_predict(model, xc_va, xn_va))
        if auc > best_auc:
            best_auc, best_state = auc, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return nn_predict(model, xc_va, xn_va), nn_predict(model, to(NN_CAT_TE), to(NN_NUM_TE))

results['mlp'] = run_cv('mlp', fit_nn, n_seeds=1)

## 9. Ensemble
### 9.1 Compare models
OOF AUC of each model and how correlated their predictions are — lower correlation means more to gain from blending.

In [ ]:
names = list(results)
display(pd.Series({n: results[n]['score'] for n in names}, name='OOF AUC').sort_values(ascending=False).round(5).to_frame())
display(pd.DataFrame({n: results[n]['oof'] for n in names}).corr(method='spearman').round(4))

### 9.2 Stacking vs. rank averaging
* **Stacking** — a logistic regression on the *logits* of every model's OOF predictions. Its own score is measured with
  5-fold `cross_val_predict`, so it is not flattered by fitting and scoring on the same rows.
* **Rank average** — equal-weight average of each model's ranks (a robust baseline for AUC).

The better of the two by OOF AUC is used for the submission.

In [ ]:
clip = lambda p: np.clip(p, 1e-6, 1 - 1e-6)
S_oof = np.column_stack([logit(clip(results[n]['oof'])) for n in names])
S_test = np.column_stack([logit(clip(results[n]['pred'])) for n in names])

stacker = LogisticRegression(C=1.0, max_iter=1000)
stack_oof = cross_val_predict(stacker, S_oof, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                              method='predict_proba')[:, 1]
stack_auc = roc_auc_score(y, stack_oof)
stacker.fit(S_oof, y)
print('Stacker weights:', dict(zip(names, stacker.coef_[0].round(3))))

rank = lambda M: np.column_stack([rankdata(c) / len(c) for c in M.T])
rank_auc = roc_auc_score(y, rank(S_oof).mean(axis=1))

best_single = max(names, key=lambda n: results[n]['score'])
print(f'Best single ({best_single}): {results[best_single]["score"]:.5f}')
print(f'Rank average:  {rank_auc:.5f}')
print(f'Stacking (LR): {stack_auc:.5f}')

if stack_auc >= rank_auc:
    final_pred, method = stacker.predict_proba(S_test)[:, 1], 'stacking'
else:
    final_pred, method = rank(S_test).mean(axis=1), 'rank average'
print('Using:', method)

## 10. Submission
Write `submission.csv` (row order taken from `sample_submission.csv`). Then either click **Submit** from the notebook's
*Output* tab, or *Save Version → Save & Run All* and submit that version from the competition's *Submit Prediction* page.

In [ ]:
sub = pd.DataFrame({'id': test['id'].values, TARGET: final_pred})
sub = sample[['id']].merge(sub, on='id', how='left')
assert sub[TARGET].notna().all() and len(sub) == len(sample)
sub.to_csv(f'{OUT_DIR}/submission.csv', index=False)
print(f'Saved {OUT_DIR}/submission.csv  shape={sub.shape}')
print(sub[TARGET].describe().round(4))
sub.head()